# B2-023 — Book 1 and Book 2 Bridge Diagnostic

*30 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

This 30-minute diagnostic checks the Book 1 tools and the B2-019–B2-022 concepts that B2-023 imports without re-teaching.
Answer both checkpoints in each section before reading its collected answers.
If a section is not fluent, follow the named remediation link before Session 1.
The sigmoid, binary cross-entropy, KL for densities, the Jensen–Shannon divergence, pointwise optimization, the sum rule for independent Gaussians, the diffusion posterior, classifier-free guidance, and cross-attention conditioning are **not** assumed; Sessions 1–5 teach them.

## 1. Arrays, broadcasting, and seeded randomness

Every computation in this unit is batched: a batch of 2-D points is a `(B,2)` tensor with one row per example.
Broadcasting lets a `(B,1)` column of per-row coefficients multiply a `(B,2)` batch row by row, which is exactly how the diffusion coefficients $\sqrt{\bar\alpha_t}$ are applied (Session 3).
Every random draw comes from a dedicated, seeded `torch.Generator` that is created once and passed around; two runs with the same seeds and the same sequence of draws produce identical numbers.
Remediation: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb) for `numpy-arrays`, `broadcasting`, and `random-seeding`.

**Checkpoint 1A.** For `c` of shape `(5,1)` and `x` of shape `(5,2)`, what is the shape of `c * x`, and what multiplies `x[3,1]`?

**Checkpoint 1B.** A training loop draws `t` and then `eps` from one generator. A teammate swaps the order. Will their run reproduce yours?

**Collected answers.** `(5,2)`, and `x[3,1]` is multiplied by `c[3,0]`. No: the generator's stream is consumed in a different order, so both `t` and `eps` take different values.

## 2. Matrix multiplication and orthonormal rows

`A @ B` contracts the last axis of `A` with the first axis of `B`: `(B,4) @ (4,2)` is `(B,2)`.
A matrix $E$ of shape $(2,4)$ has **orthonormal rows** when $EE^\top=I_2$: each row has length $1$ and the rows are perpendicular.
Session 5 uses such an $E$ as a frozen encoder, `encode(x) = x @ E.T`, and its transpose as the decoder, `decode(z) = z @ E`.
Remediation: [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb) for `matrix-multiplication`; [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb) for linear maps used as models.

**Checkpoint 2A.** What is the shape of `x @ E.T` for `x` of shape `(192,4)` and `E` of shape `(2,4)`?

**Checkpoint 2B.** Check that the rows $(\tfrac12,\tfrac12,\tfrac12,\tfrac12)$ and $(\tfrac12,-\tfrac12,-\tfrac12,\tfrac12)$ are orthonormal.

**Collected answers.** `(192,2)`. Each has squared length $4\cdot\frac14=1$, and their dot product is $\frac14-\frac14-\frac14+\frac14=0$.

## 3. Expectation, variance, independence, and covariance

Book 1 F5 defined $E[X]$, $\operatorname{Var}(X)=E[(X-EX)^2]$, the `covariance` $\operatorname{Cov}(X,Y)=E[(X-EX)(Y-EY)]$, `independence`, and the `variance-of-sums` rule
$$\operatorname{Var}(X+Y)=\operatorname{Var}X+\operatorname{Var}Y+2\operatorname{Cov}(X,Y),$$
together with $\operatorname{Var}(cX)=c^2\operatorname{Var}X$ and the fact that independent variables have covariance $0$.
Session 3 combines these with one stated fact (Gaussianity of sums) to collapse many noise steps into one.
Remediation: [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb) for `expectation`, `variance`, `independence`, `variance-of-sums`, and `covariance`.

**Checkpoint 3A.** $X$ and $Y$ are independent with variances $0.64$ and $0.36$. What is $\operatorname{Var}(X+Y)$?

**Checkpoint 3B.** What is $\operatorname{Var}(0.8X)$ if $\operatorname{Var}X=1$?

**Collected answers.** $0.64+0.36=1$, because independence makes the covariance term vanish. $0.64$.

## 4. Gaussians and simulation

The univariate `gaussian-distribution` $N(\mu,\sigma^2)$ has mean $\mu$ and variance $\sigma^2$; $\mu+\sigma\varepsilon$ with $\varepsilon\sim N(0,1)$ is a draw from it.
Monte Carlo (`sampling-simulation`) estimates a mean or variance from many seeded draws; its error shrinks like $1/\sqrt n$.
Remediation: [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb) for `gaussian-distribution` and `sampling-simulation`.

**Checkpoint 4A.** If $\varepsilon\sim N(0,1)$, what is the distribution of $3+0.5\varepsilon$?

**Checkpoint 4B.** A Monte Carlo variance from $400{,}000$ draws of a unit-variance variable is $1.003$. Is that evidence of a bug?

**Collected answers.** $N(3,0.25)$. No: the sampling error of the variance estimate is of order $\sqrt{2/n}\approx0.002$, so $0.003$ is ordinary fluctuation.

## 5. Gradients, MLPs, and the PyTorch training loop

An MLP alternates `nn.Linear` layers with ReLU (`relu-activation`, `mlp-architecture`); an `nn.Module` registers its parameters; tensors that require gradients (`requires-grad`) let autograd compute a `gradient` of a scalar loss such as the `mse-loss`.
One update is `optimizer.zero_grad(set_to_none=True)` → forward → loss → `loss.backward()` → `optimizer.step()` with `torch.optim.Adam` (`torch-optimizers`, `autograd-training`).
Train/test discipline (`train-test-split`): only training rows ever enter a training forward pass, `backward()`, or an optimizer step.
Remediation: [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb) for `gradient`; [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb) for `train-test-split`; [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb) for `mse-loss`; [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb) for `relu-activation` and `mlp-architecture`; [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb) for `torch-tensors`, `nn-module`, and `requires-grad`; [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb) for `torch-optimizers` and `autograd-training`.

**Checkpoint 5A.** How many parameters does `nn.Sequential(nn.Linear(3,64), nn.ReLU(), nn.Linear(64,64), nn.ReLU(), nn.Linear(64,2))` have?

**Checkpoint 5B.** An optimizer was built from network A's parameters only. Can `optimizer.step()` change network B?

**Collected answers.** $(3\cdot64+64)+(64\cdot64+64)+(64\cdot2+2)=256+4160+130=4546$. No: an optimizer only updates the parameter tensors it was given, whatever gradients other tensors hold.

## 6. Softmax and cross-entropy

For logits $\ell_1,\dots,\ell_C$, `softmax` gives probabilities $e^{\ell_c}/\sum_ke^{\ell_k}$, and the `cross-entropy-loss` for the true class $y$ is $-\log$ of its probability.
Session 1 specializes both to two classes (real versus fake), which produces the sigmoid and binary cross-entropy; Session 5 uses softmax over attention scores.
Remediation: [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb) for `softmax` and `cross-entropy-loss`.

**Checkpoint 6A.** Compute the softmax of the logits $(\ln3,0)$.

**Checkpoint 6B.** What is the cross-entropy when the true class has probability $\tfrac34$?

**Collected answers.** $(\frac{3}{4},\frac14)$. $-\log\frac34=0.287682$.

## 7. Book 2 imports: attention, embeddings, U-Net, Gaussians, KL, and VAEs

B2-023 composes four earlier Book 2 units; it uses their concepts without re-teaching them.

* [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb): `query-key-value-attention`, scores $QK^\top/\sqrt d$ and a softmax over keys (Session 5's cross-attention).
* [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb): `learned-token-embedding`, an `nn.Embedding` lookup table trained with the model (Session 5's class tokens).
* [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb): the `unet` encoder–decoder with skip connections, Stable Diffusion's denoiser, and the modal-ownership rule for cross-attention (Session 5).
* [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb): the `multivariate-gaussian`, `gaussian-reparameterization` $z=\mu+\sigma\odot\varepsilon$ with the noise passed in, the `kl-divergence` and its Gaussian closed forms, the `autoencoder`, and the `variational-autoencoder` with its ELBO (Sessions 1, 3, 4, 5).

**Checkpoint 7A.** In B2-021's ownership rule, which modality supplies the queries when image tokens read from text tokens?

**Checkpoint 7B.** Write the KL from $N(\mu,\sigma^2)$ to $N(\nu,\tau^2)$ in one dimension (B2-022).

**Collected answers.** The image tokens (the modality being updated) supply the queries; the text tokens supply keys and values. $\log\frac{\tau}{\sigma}+\frac{\sigma^2+(\mu-\nu)^2}{2\tau^2}-\frac12$.

## 8. Readiness and explicit remediation

Proceed to Session 1 only if every checkpoint above was fluent.
Otherwise use the link for the exact gap; do not guess through a missing prerequisite.
The unit's own new tools are taught from scratch where they are first used: the sigmoid, BCE with logits, continuous KL and Jensen–Shannon, and pointwise optimization (Session 1); the sum rule for independent Gaussians (Session 3); the stated diffusion posterior and the equal-variance KL argument (Session 4); conditioning dropout, classifier-free guidance, and cross-attention conditioning (Session 5).

**Checkpoint 8A.** Which link do you follow if you cannot explain why held-out rows must never enter `backward()`?

**Checkpoint 8B.** Do you need to know the sigmoid before Session 1?

**Collected answers.** [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb) for the train-test split, then C11 for the training loop. No: Session 1 defines it and derives every property the unit uses.

In [ ]:
import torch

readiness = {
    "broadcast a (5,1) column": (torch.ones(5, 1) * torch.ones(5, 2)).shape == (5, 2),
    "matrix product shape": (torch.ones(192, 4) @ torch.ones(2, 4).T).shape == (192, 2),
    "softmax of (ln 3, 0)": torch.allclose(torch.softmax(torch.tensor([torch.log(torch.tensor(3.0)).item(), 0.0]), dim=0),
                                           torch.tensor([0.75, 0.25]), atol=1e-6, rtol=1e-6),
    "MLP parameter count": sum(p.numel() for p in torch.nn.Sequential(torch.nn.Linear(3, 64), torch.nn.ReLU(),
                                                                       torch.nn.Linear(64, 64), torch.nn.ReLU(),
                                                                       torch.nn.Linear(64, 2)).parameters()) == 4546,
}
print(readiness)
bridge_feedback = "ready for B2-023" if all(readiness.values()) else "follow the remediation links first"
print(bridge_feedback)